In [ ]:
from web3 import Web3

# Setup Web3 provider
INFURA_URL = "https://mainnet.infura.io/v3/e0131e1abdf643bbb0b9d900ad2c1de3" # Nickys API Key (use if you want or make your own)
web3 = Web3(Web3.HTTPProvider(INFURA_URL))

# Uniswap V3 ETH/USDC 0.3% pool address
POOL_ADDRESS = Web3.to_checksum_address("0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8")

# Common token decimals
TOKEN_DECIMALS = {
    "0xA0b86991c6218b36c1d19d4a2e9eb0ce3606eb48".lower(): 6,   # USDC
    "0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2".lower(): 18   # WETH
}

token_to_string = {
    "0xA0b86991c6218b36c1d19d4a2e9eb0ce3606eb48".lower(): "USDC",
    "0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2".lower(): "WETH"
}

# Minimal ABI for token0 and token1
POOL_META_ABI = [
    {"constant": True, "inputs": [], "name": "token0", "outputs": [{"name": "", "type": "address"}], "type": "function"},
    {"constant": True, "inputs": [], "name": "token1", "outputs": [{"name": "", "type": "address"}], "type": "function"}
]

# Swap event ABI
SWAP_EVENT_ABI = [{
    "anonymous": False,
    "inputs": [
        {"indexed": True, "internalType": "address", "name": "sender", "type": "address"},
        {"indexed": True, "internalType": "address", "name": "recipient", "type": "address"},
        {"indexed": False, "internalType": "int256", "name": "amount0", "type": "int256"},
        {"indexed": False, "internalType": "int256", "name": "amount1", "type": "int256"},
        {"indexed": False, "internalType": "uint160", "name": "sqrtPriceX96", "type": "uint160"},
        {"indexed": False, "internalType": "uint128", "name": "liquidity", "type": "uint128"},
        {"indexed": False, "internalType": "int24", "name": "tick", "type": "int24"}
    ],
    "name": "Swap",
    "type": "event"
}]

def get_token_decimals(address):
    return TOKEN_DECIMALS.get(address.lower(), 18)  # Default to 18

def decode_swap_logs(pool_address, from_block, to_block="latest"):
    topic0 ="0x" + web3.keccak(text="Swap(address,address,int256,int256,uint160,uint128,int24)").hex()
    
    # Get token0 and token1
    meta_contract = web3.eth.contract(address=pool_address, abi=POOL_META_ABI)
    token0 = meta_contract.functions.token0().call()
    token1 = meta_contract.functions.token1().call()

    dec0 = get_token_decimals(token0)
    dec1 = get_token_decimals(token1)
    
    
    token0_str = token_to_string.get(token0.lower(), "Unknown")
    token1_str = token_to_string.get(token1.lower(), "Unknown")

    contract = web3.eth.contract(address=pool_address, abi=SWAP_EVENT_ABI)
    
    logs = web3.eth.get_logs({
        "fromBlock": from_block,
        "toBlock": to_block,
        "address": pool_address,
        "topics": [topic0]
    })

    print(f"🧾 Found {len(logs)} swaps")
    
    for log in logs:
        decoded = contract.events.Swap().process_log(log)
        args = decoded["args"]

        print("—" * 50)
        print(f"Block: {log['blockNumber']}")
        print(f"Sender: {args['sender']}")
        print(f"Recipient: {args['recipient']}")
        print(f"Token0 ({token0}: {token0_str}): {args['amount0'] / (10 ** dec0)}")
        print(f"Token1 ({token1}: {token1_str})): {args['amount1'] / (10 ** dec1)}")
        print(f"Tick: {args['tick']}")
        print(f"SqrtPriceX96: {args['sqrtPriceX96']}")

# === Call it ===
latest = web3.eth.block_number
decode_swap_logs(POOL_ADDRESS, latest - 1000, latest)


🧾 Found 66 swaps
——————————————————————————————————————————————————
Block: 22383809
Sender: 0x06CFf7088619C7178F5e14f0B119458d08d2f5ef
Recipient: 0x06CFf7088619C7178F5e14f0B119458d08d2f5ef
Token0 (0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48: USDC): 0.256934
Token1 (0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2: WETH)): -0.000143690472288846
Tick: 201461
SqrtPriceX96: 1876442968249937103936417568846040
——————————————————————————————————————————————————
Block: 22383851
Sender: 0x0DDC6F9CE13b985DFd730b8048014B342D1b54F7
Recipient: 0x0DDC6F9CE13b985DFd730b8048014B342D1b54F7
Token0 (0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48: USDC): 21.46596
Token1 (0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2: WETH)): -0.012004857105861973
Tick: 201461
SqrtPriceX96: 1876442762824562027790607421487778
——————————————————————————————————————————————————
Block: 22383943
Sender: 0x51C72848c68a965f66FA7a88855F9f7784502a7F
Recipient: 0x51C72848c68a965f66FA7a88855F9f7784502a7F
Token0 (0xA0b86991c6218b36c1d19D4a2e9Eb0cE3